In [27]:
#imports
import pandas as pd
from pathlib import Path
import plotly.express as px
from DataLoader import DataLoader
root = r"C:\Users\20202310\Desktop\MSc scriptie\MSc_Graduation_Project\data\LUNDPROBE\ExtendedSamples\development"

rootpath = Path(root)
subjects = sorted([p.name for p in rootpath.iterdir() if p.is_dir()])
print(subjects)

folder_to_analyse = Path(r"C:\Users\20202310\Desktop\MSc scriptie\MSc_Graduation_Project\fusion_evaluation\Against_GT\Volume results")

['newAcq_050f229dc2bdb64c', 'newAcq_0b4940fa31a1d650', 'newAcq_0cc559a8bd82a14a', 'newAcq_1b911d6cb2348f30', 'newAcq_1e0f8b9b01ce5f0b', 'newAcq_250d6075dd465a1a', 'newAcq_433a8d44fddd5b7f', 'newAcq_47ceabdbca398517', 'newAcq_486b7494ee9d71e7', 'newAcq_4a136e8fe320bd13']


In [28]:
#LOAD AVAILABLE DATA

#E_out_consensus_crossvalidation

folder = folder_to_analyse

csv_dataframes = {}

for csv_file in folder.glob("*.csv"):
    df_name = csv_file.stem
    csv_dataframes[df_name] = pd.read_csv(csv_file)

print(f"Loaded {len(csv_dataframes)} CSV files:")
for name, df in csv_dataframes.items():
    print(f"{name}: {df.shape}")

# Add method name from CSV filename
all_data = []

for method_name, df in csv_dataframes.items():
    temp = df.copy()
    temp["method"] = method_name
    all_data.append(temp)

long_df = pd.concat(all_data, ignore_index=True)

long_df.head()


id_col = "subject name"      # change if needed
method_col = "method"   # change if needed

metric_columns = [
    col for col in long_df.select_dtypes(include="number").columns
    if col not in [id_col]
]

method_options = sorted(long_df[method_col].unique())

Loaded 16 CSV files:
nnUnet_volume: (10, 13)
obsB_volume: (10, 13)
obsC_volume: (10, 13)
obsD_volume: (10, 13)
obsE_volume: (10, 13)
volume_results_Staple_0.0_bbox_1.0: (10, 16)
volume_results_Staple_0.1_bbox_0.9: (10, 16)
volume_results_Staple_0.2_bbox_0.8: (10, 16)
volume_results_Staple_0.3_bbox_0.7: (10, 16)
volume_results_Staple_0.4_bbox_0.6: (10, 16)
volume_results_Staple_0.5_bbox_0.5: (10, 16)
volume_results_Staple_0.6_bbox_0.4: (10, 16)
volume_results_Staple_0.7_bbox_0.3: (10, 16)
volume_results_Staple_0.8_bbox_0.2: (10, 16)
volume_results_Staple_0.9_bbox_0.1: (10, 16)
volume_results_Staple_1.0_bbox_0.0: (10, 16)


In [30]:
# INTERACTIVE DESCRIPTIVE STATISTICS FOR ALL METRICS

import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

def compute_descriptive_statistics(selected_metrics, selected_methods, round_decimals=3):

    selected_metrics = list(selected_metrics)
    selected_methods = list(selected_methods)

    df = long_df[long_df[method_col].isin(selected_methods)].copy()

    all_stats = []

    for metric in selected_metrics:

        paired_df = df.pivot(
            index=id_col,
            columns=method_col,
            values=metric
        ).dropna()

        if paired_df.empty:
            continue

        stats_df = paired_df.describe().T

        stats_df["median"] = paired_df.median()
        stats_df["iqr"] = paired_df.quantile(0.75) - paired_df.quantile(0.25)
        stats_df["missing_values"] = paired_df.isna().sum()
        stats_df["metric"] = metric

        all_stats.append(stats_df)

    if len(all_stats) == 0:
        print("No valid paired data found.")
        return

    final_stats = pd.concat(all_stats)

    final_stats = final_stats.reset_index().rename(
        columns={"index": "method"}
    )

    final_stats = final_stats[
        [
            "metric",
            "method",
            "count",
            "mean",
            "std",
            "min",
            "25%",
            "median",
            "75%",
            "iqr",
            "max",
            "missing_values"
        ]
    ]

    display(final_stats.round(round_decimals))


# Automatically get numeric metric columns
available_metrics = [
    col for col in long_df.select_dtypes(include="number").columns
    if col != id_col
]

metric_selector = widgets.SelectMultiple(
    options=available_metrics,
    value=tuple(available_metrics),
    description="Metrics:",
    rows=min(10, len(available_metrics))
)

method_selector = widgets.SelectMultiple(
    options=method_options,
    value=tuple(method_options),
    description="Methods:",
    rows=min(8, len(method_options))
)

round_slider = widgets.IntSlider(
    value=3,
    min=0,
    max=6,
    step=1,
    description="Decimals:"
)

out = widgets.interactive_output(
    compute_descriptive_statistics,
    {
        "selected_metrics": metric_selector,
        "selected_methods": method_selector,
        "round_decimals": round_slider
    }
)

display(
    widgets.VBox([
        metric_selector,
        method_selector,
        round_slider
    ]),
    out
)

Output()

In [31]:
# INTERACTIVE PLOTLY DASHBOARD
def analyze_metric_plotly(
    metric,
    selected_methods,
    show_boxplot,
    show_paired_lines,
):

    selected_methods = list(selected_methods)

    if len(selected_methods) < 2:
        print("Select at least two methods.")
        return

    # Filter methods
    df = long_df[long_df[method_col].isin(selected_methods)].copy()

    # Create paired dataframe
    paired_df = df.pivot(
        index=id_col,
        columns=method_col,
        values=metric
    )

    paired_df = paired_df.dropna()

    print(f"\nMetric: {metric}")
    print(f"Number of paired samples: {len(paired_df)}")

    # Reset subject numbering
    paired_df = paired_df.reset_index(drop=True)

    # ---------------- BOXPLOT ----------------

    if show_boxplot:

        box_df = paired_df.melt(
            var_name="Method",
            value_name=metric
        )

        fig = px.box(
            box_df,
            x="Method",
            y=metric,
            points="all",
            title=f"Paired comparison of {metric}"
        )

        fig.update_layout(
            height=500,
            width=900
        )

        fig.show()

    # ---------------- PAIRED LINE PLOT ----------------

    if show_paired_lines:

        fig = px.line(
            paired_df,
            markers=True,
            labels={
                "index": "Subject Number",
                "value": metric,
                "variable": "Method"
            },
            title=f"{metric} per subject"
        )

        fig.update_layout(
            xaxis_title="Subject Number",
            yaxis_title=metric,
            height=600,
            width=1000
        )

        fig.show()


# ---------------- WIDGETS ----------------

metric_dropdown = widgets.Dropdown(
    options=metric_columns,
    description="Metric:"
)

methods_select = widgets.SelectMultiple(
    options=method_options,
    value=tuple(method_options),
    description="Methods:"
)

boxplot_checkbox = widgets.Checkbox(
    value=True,
    description="Boxplot"
)

paired_checkbox = widgets.Checkbox(
    value=True,
    description="Paired lines"
)


ui = widgets.VBox([
    metric_dropdown,
    methods_select,
    boxplot_checkbox,
    paired_checkbox,
])

out = widgets.interactive_output(
    analyze_metric_plotly,
    {
        "metric": metric_dropdown,
        "selected_methods": methods_select,
        "show_boxplot": boxplot_checkbox,
        "show_paired_lines": paired_checkbox,
    }
)

display(ui, out)

Output()

In [19]:
# CELL TO CREATE INTERACTIVE TABLE TO EXPLORE SLICE-BASED RESULTS
# INPUT CAN BE:
#   1. A single CSV file
#   2. A folder containing multiple CSV files

from pathlib import Path

import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output


def explore_slice_results(path, recursive=False):
    """
    Interactively explore numeric results from either:
        - one CSV file, or
        - all CSV files inside a folder.

    Parameters
    ----------
    path : str or pathlib.Path
        Path to a CSV file or a folder containing CSV files.

    recursive : bool, default=False
        If True, also search for CSV files in subfolders.

    Notes
    -----
    When a folder is provided, the following columns are added:
        - source_file: CSV filename without extension
        - source_filename: complete CSV filename
        - source_path: complete CSV path

    Existing columns with these names are preserved using an
    automatically generated alternative name.
    """

    path = Path(path)

    # ============================================================
    # Load data
    # ============================================================

    if not path.exists():
        raise FileNotFoundError(f"Path does not exist:\n{path}")

    if path.is_file():
        if path.suffix.lower() != ".csv":
            raise ValueError(f"Input file must be a CSV file:\n{path}")

        csv_files = [path]

    elif path.is_dir():
        search_pattern = "**/*.csv" if recursive else "*.csv"
        csv_files = sorted(path.glob(search_pattern))

        if len(csv_files) == 0:
            raise FileNotFoundError(
                f"No CSV files found in:\n{path}\n"
                f"recursive={recursive}"
            )

    else:
        raise ValueError(f"Input must be a CSV file or folder:\n{path}")

    loaded_dataframes = []
    failed_files = []

    for csv_file in csv_files:
        try:
            temp_df = pd.read_csv(csv_file)
        except Exception as error:
            failed_files.append((csv_file.name, str(error)))
            continue

        # Avoid overwriting columns already present in the CSV
        source_file_col = "source_file"
        source_filename_col = "source_filename"
        source_path_col = "source_path"

        while source_file_col in temp_df.columns:
            source_file_col = "_" + source_file_col

        while source_filename_col in temp_df.columns:
            source_filename_col = "_" + source_filename_col

        while source_path_col in temp_df.columns:
            source_path_col = "_" + source_path_col

        temp_df[source_file_col] = csv_file.stem
        temp_df[source_filename_col] = csv_file.name
        temp_df[source_path_col] = str(csv_file)

        loaded_dataframes.append(temp_df)

    if len(loaded_dataframes) == 0:
        error_text = "\n".join(
            f"- {filename}: {error}"
            for filename, error in failed_files
        )

        raise RuntimeError(
            "None of the CSV files could be loaded.\n"
            f"{error_text}"
        )

    # pd.concat automatically handles CSVs with different columns.
    # Missing columns in individual files become NaN.
    df = pd.concat(
        loaded_dataframes,
        ignore_index=True,
        sort=False
    )

    # Find the actual added source column names
    source_columns = [
        col
        for col in df.columns
        if col.lstrip("_") in {
            "source_file",
            "source_filename",
            "source_path",
        }
    ]

    preferred_source_col = next(
        (
            col for col in source_columns
            if col.lstrip("_") == "source_file"
        ),
        None
    )

    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
    categorical_cols = [
        col for col in df.columns
        if col not in numeric_cols
    ]

    if len(numeric_cols) == 0:
        raise ValueError(
            "No numeric columns were found in the loaded CSV files."
        )

    print(
        f"Loaded {len(loaded_dataframes)} of {len(csv_files)} CSV files "
        f"with {len(df):,} total rows."
    )

    if failed_files:
        print("\nFiles that could not be loaded:")
        for filename, error in failed_files:
            print(f"  - {filename}: {error}")

    # ============================================================
    # Widgets
    # ============================================================

    metric_select = widgets.SelectMultiple(
        options=numeric_cols,
        value=tuple(numeric_cols[:min(5, len(numeric_cols))]),
        description="Metrics:",
        rows=10,
        layout=widgets.Layout(width="400px")
    )

    default_group = (
        "method"
        if "method" in categorical_cols
        else preferred_source_col
        if preferred_source_col in categorical_cols
        else "None"
    )

    group_dropdown = widgets.Dropdown(
        options=["None"] + categorical_cols,
        value=default_group,
        description="Group:",
        layout=widgets.Layout(width="450px")
    )

    group_values_select = widgets.SelectMultiple(
        options=[],
        value=(),
        description="Show:",
        rows=8,
        layout=widgets.Layout(width="450px")
    )

    stat_select = widgets.SelectMultiple(
        options=[
            "count",
            "mean",
            "std",
            "median",
            "min",
            "max",
            "missing",
            "missing_percent",
        ],
        value=("count", "mean", "std", "median", "min", "max"),
        description="Stats:",
        rows=10,
        layout=widgets.Layout(width="250px")
    )

    round_decimals = widgets.BoundedIntText(
        value=4,
        min=0,
        max=10,
        step=1,
        description="Decimals:",
        layout=widgets.Layout(width="180px")
    )

    row_count_label = widgets.HTML()

    out = widgets.Output()

    # ============================================================
    # Helper functions
    # ============================================================

    def safely_sorted_unique(series):
        """
        Return unique non-missing values as strings.

        Sorting as strings avoids errors when a column contains a mixture
        of strings, integers, booleans, etc.
        """
        return sorted(
            series.dropna().astype(str).unique().tolist(),
            key=str.lower
        )

    def update_group_values(change=None):
        group_col = group_dropdown.value

        if group_col == "None":
            group_values_select.options = []
            group_values_select.value = ()
            group_values_select.disabled = True
        else:
            values = safely_sorted_unique(df[group_col])

            group_values_select.options = values
            group_values_select.value = tuple(values)
            group_values_select.disabled = False

        update_table()

    def get_filtered_df():
        group_col = group_dropdown.value

        if group_col == "None":
            return df.copy()

        selected_groups = list(group_values_select.value)

        if len(selected_groups) == 0:
            return df.iloc[0:0].copy()

        return df[
            df[group_col].astype(str).isin(selected_groups)
        ].copy()

    def summarize_numeric(data, selected_metrics, selected_stats):
        rows = []

        for metric in selected_metrics:
            values = pd.to_numeric(data[metric], errors="coerce")

            row = {"metric": metric}

            if "count" in selected_stats:
                row["count"] = values.count()

            if "mean" in selected_stats:
                row["mean"] = values.mean()

            if "std" in selected_stats:
                row["std"] = values.std()

            if "median" in selected_stats:
                row["median"] = values.median()

            if "min" in selected_stats:
                row["min"] = values.min()

            if "max" in selected_stats:
                row["max"] = values.max()

            if "missing" in selected_stats:
                row["missing"] = values.isna().sum()

            if "missing_percent" in selected_stats:
                row["missing_percent"] = 100 * values.isna().mean()

            rows.append(row)

        return pd.DataFrame(rows)

    def update_table(change=None):
        with out:
            clear_output(wait=True)

            plot_df = get_filtered_df()
            selected_metrics = list(metric_select.value)
            selected_stats = list(stat_select.value)
            group_col = group_dropdown.value

            row_count_label.value = (
                f"<b>Selected rows:</b> {len(plot_df):,} / {len(df):,}"
            )

            if plot_df.empty:
                print("No data selected.")
                return

            if len(selected_metrics) == 0:
                print("Select at least one metric.")
                return

            if len(selected_stats) == 0:
                print("Select at least one statistic.")
                return

            if group_col == "None":
                summary_df = summarize_numeric(
                    plot_df,
                    selected_metrics,
                    selected_stats
                )

            else:
                summaries = []

                # dropna=False includes rows where the grouping value is missing
                for group_name, group_df in plot_df.groupby(
                    group_col,
                    dropna=False,
                    sort=False
                ):
                    group_summary = summarize_numeric(
                        group_df,
                        selected_metrics,
                        selected_stats
                    )

                    group_summary.insert(
                        0,
                        group_col,
                        group_name
                    )

                    summaries.append(group_summary)

                if len(summaries) == 0:
                    print("No groups available.")
                    return

                summary_df = pd.concat(
                    summaries,
                    ignore_index=True
                )

            summary_df = summary_df.round(round_decimals.value)

            display(summary_df)

    # ============================================================
    # Connect widgets
    # ============================================================

    metric_select.observe(update_table, names="value")
    stat_select.observe(update_table, names="value")
    group_dropdown.observe(update_group_values, names="value")
    group_values_select.observe(update_table, names="value")
    round_decimals.observe(update_table, names="value")

    # ============================================================
    # Display
    # ============================================================

    display(
        widgets.VBox([
            widgets.HBox([
                group_dropdown,
                round_decimals
            ]),
            group_values_select,
            row_count_label,
            widgets.HBox([
                metric_select,
                stat_select
            ]),
            out
        ])
    )

    update_group_values()

    # Returning the dataframe is useful if you want to inspect it afterward
    return df

In [22]:
results_df = explore_slice_results(
    r"C:\Users\20202310\Desktop\MSc scriptie\MSc_Graduation_Project\fusion_evaluation\Against_GT\Slice results"
)

Loaded 16 of 16 CSV files with 3,308 total rows.


In [26]:
# CELL TO INTERACTIVELY COMPARE THE SAME SLICES ACROSS WEIGHTINGS

from pathlib import Path
import re

import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
from pandas.errors import EmptyDataError


def compare_weightings_same_slices(
    path,
    metric="SurfaceDice@1.0mm",
    recursive=False,
):
    """
    Interactively compare slice-based metrics across fusion weightings,
    observers, and nnUNet.

    Weighted fusion CSVs are recognized through the StapleWeight and
    BBoxWeight columns. Observer B-E and nnUNet CSVs are recognized from
    either a method/model/observer column or their filenames.

    Subject-slice pairs are aligned across all selected methods. Only rows
    with a valid value for every selected method are shown.
    """

    path = Path(path)
    decimals = 4

    # ============================================================
    # Load CSV files
    # ============================================================

    if not path.exists():
        raise FileNotFoundError(f"Path does not exist:\n{path}")

    if path.is_file():
        if path.suffix.lower() != ".csv":
            raise ValueError("The selected file is not a CSV file.")

        csv_files = [path]

    elif path.is_dir():
        pattern = "**/*.csv" if recursive else "*.csv"
        csv_files = sorted(path.glob(pattern))

        if not csv_files:
            raise FileNotFoundError(f"No CSV files found in:\n{path}")

    else:
        raise ValueError("Input must be a CSV file or folder.")

    dataframes = []
    skipped_files = []

    for csv_file in csv_files:
        try:
            temp_df = pd.read_csv(csv_file)
        except EmptyDataError:
            skipped_files.append((csv_file.name, "completely empty"))
            continue
        except Exception as error:
            skipped_files.append((csv_file.name, str(error)))
            continue

        if temp_df.empty:
            skipped_files.append((csv_file.name, "no data rows"))
            continue

        temp_df["source_file"] = csv_file.stem
        dataframes.append(temp_df)

    if skipped_files:
        print("Skipped CSV files:")
        for filename, reason in skipped_files:
            print(f"  {filename}: {reason}")

    if not dataframes:
        raise ValueError("None of the CSV files contained readable data.")

    df = pd.concat(
        dataframes,
        ignore_index=True,
        sort=False,
    )

    # ============================================================
    # Detect important columns
    # ============================================================

    subject_candidates = [
        "subject",
        "subject_name",
        "subject name",
        "Subject",
        "SubjectName",
    ]

    slice_candidates = [
        "slice",
        "slice_index",
        "slice_idx",
        "slice_nr",
        "slice_number",
        "z",
        "z_index",
        "image_slice",
        "absolute_slice_index",
    ]

    subject_col = next(
        (column for column in subject_candidates if column in df.columns),
        None,
    )

    slice_col = next(
        (column for column in slice_candidates if column in df.columns),
        None,
    )

    if subject_col is None:
        raise ValueError(
            "Could not detect the subject column.\n"
            f"Available columns:\n{df.columns.tolist()}"
        )

    if slice_col is None:
        raise ValueError(
            "Could not detect the slice-index column.\n"
            f"Available columns:\n{df.columns.tolist()}"
        )

    # ============================================================
    # Create one readable comparison label per row
    # ============================================================

    def prettify_method_name(value, column_name=None):
        """Turn common observer/model names into consistent display labels."""

        text = str(value).strip()
        compact = re.sub(r"[^a-z0-9]+", "", text.lower())

        if "nnunet" in compact:
            return "nnUNet"

        observer_match = re.search(
            r"(?:observer|obs)[_\-\s]*([bcde])",
            text,
            flags=re.IGNORECASE,
        )

        if observer_match:
            return f"Observer {observer_match.group(1).upper()}"

        if (
            column_name is not None
            and "observer" in column_name.lower()
            and compact in {"b", "c", "d", "e"}
        ):
            return f"Observer {compact.upper()}"

        # Make an otherwise unknown filename a little easier to read.
        text = re.sub(
            r"^(slice[_\-\s]*results?|results?[_\-\s]*slice)[_\-\s]*",
            "",
            text,
            flags=re.IGNORECASE,
        )
        return text.replace("_", " ").strip()

    explicit_method_columns = [
        column
        for column in [
            "comparison_method",
            "result_name",
            "prediction_name",
            "model",
            "observer",
            "Observer",
        ]
        if column in df.columns
    ]

    def determine_comparison_method(row):
        # Fusion result: use its two weights.
        if (
            "StapleWeight" in df.columns
            and "BBoxWeight" in df.columns
            and pd.notna(row.get("StapleWeight"))
            and pd.notna(row.get("BBoxWeight"))
        ):
            return (
                f"Staple {row['StapleWeight']:g} | "
                f"BBox {row['BBoxWeight']:g}"
            )

        # Existing masks: first try a descriptive column in the CSV.
        for column in explicit_method_columns:
            value = row.get(column)
            if pd.notna(value) and str(value).strip():
                return prettify_method_name(value, column)

        # Otherwise infer Observer B-E or nnUNet from the CSV filename.
        return prettify_method_name(row["source_file"])

    df["comparison_method"] = df.apply(
        determine_comparison_method,
        axis=1,
    )

    slice_identifier_cols = [subject_col, slice_col]

    # ============================================================
    # Detect metric columns
    # ============================================================

    excluded_numeric_columns = {
        slice_col,
        "DenseWeight",
        "StapleWeight",
        "BBoxWeight",
        "Threshold",
        "LogitThreshold",
    }

    metric_options = [
        column
        for column in df.select_dtypes(include=np.number).columns
        if column not in excluded_numeric_columns
    ]

    if not metric_options:
        raise ValueError(
            "No numeric metric columns were detected.\n"
            f"Available columns:\n{df.columns.tolist()}"
        )

    if metric not in metric_options:
        metric = metric_options[0]

    # ============================================================
    # Remove rows without a usable slice index. This also safely ignores
    # whole-volume result CSVs if they are present in the same folder.
    df = df.dropna(
        subset=[subject_col, slice_col, "comparison_method"]
    ).copy()

    if df.empty:
        raise ValueError(
            "No slice-wise rows were found. Make sure 'path' points to the "
            "slice-results CSV files rather than only volume-results files."
        )

    # ============================================================
    # Detect accidental duplicate measurements
    # ============================================================

    duplicate_counts = (
        df.groupby(
            slice_identifier_cols + ["comparison_method"],
            dropna=False,
        )
        .size()
    )

    if len(duplicate_counts) > 0 and duplicate_counts.max() > 1:
        print(
            "Warning: duplicate subject-slice-method combinations were "
            "found. Their metric values will be averaged during pivoting."
        )

    # ============================================================
    # Widget options
    # ============================================================

    subjects = sorted(
        df[subject_col].dropna().astype(str).unique().tolist()
    )

    detected_methods = (
        df["comparison_method"]
        .dropna()
        .drop_duplicates()
        .tolist()
    )

    def method_sort_key(method):
        """
        Put fusion weightings first, followed by observers and nnUNet.

        This same ordering is used in the selector, summary, and table.
        """

        text = str(method)

        if text.startswith("Staple "):
            match = re.search(
                r"Staple\s+([0-9.]+)\s*\|\s*BBox\s+([0-9.]+)",
                text,
                flags=re.IGNORECASE,
            )

            if match:
                staple_weight = float(match.group(1))
                bbox_weight = float(match.group(2))
                return (0, staple_weight, bbox_weight, text.lower())

            return (0, float("inf"), float("inf"), text.lower())

        observer_match = re.fullmatch(
            r"Observer\s+([B-E])",
            text,
            flags=re.IGNORECASE,
        )

        if observer_match:
            observer_order = "BCDE".index(
                observer_match.group(1).upper()
            )
            return (1, observer_order, 0, text.lower())

        if text.lower() == "nnunet":
            return (2, 0, 0, text.lower())

        # Any other non-fusion comparison is also placed on the right.
        return (3, 0, 0, text.lower())

    method_options = sorted(
        detected_methods,
        key=method_sort_key,
    )

    if not method_options:
        raise ValueError("No comparison methods were found.")

    fusion_method_options = [
        method
        for method in method_options
        if str(method).startswith("Staple ")
    ]

    # ============================================================
    # Widgets
    # ============================================================

    subject_select = widgets.SelectMultiple(
        options=subjects,
        value=tuple(subjects),
        description="Subjects:",
        rows=min(10, max(4, len(subjects))),
        layout=widgets.Layout(width="470px"),
        style={"description_width": "initial"},
    )

    method_select = widgets.SelectMultiple(
        options=method_options,
        value=tuple(method_options),
        description="Visible columns:",
        rows=min(14, max(5, len(method_options))),
        layout=widgets.Layout(width="520px"),
        style={"description_width": "initial"},
    )

    show_all_columns_button = widgets.Button(
        description="Show all",
        tooltip="Make every comparison column visible",
        layout=widgets.Layout(width="110px"),
    )

    fusion_only_button = widgets.Button(
        description="Fusion only",
        tooltip="Show only the fusion-weighting columns",
        layout=widgets.Layout(width="110px"),
        disabled=not fusion_method_options,
    )

    column_selection_hint = widgets.HTML(
        "<small>Use Ctrl/Cmd-click to show or hide individual columns.</small>"
    )

    def show_all_columns(_):
        method_select.value = tuple(method_options)

    def show_fusion_only(_):
        method_select.value = tuple(fusion_method_options)

    show_all_columns_button.on_click(show_all_columns)
    fusion_only_button.on_click(show_fusion_only)

    metric_dropdown = widgets.Dropdown(
        options=metric_options,
        value=metric,
        description="Metric:",
        layout=widgets.Layout(width="500px"),
        style={"description_width": "initial"},
    )

    highlight_mode = widgets.ToggleButtons(
        options=[
            ("Highest", "max"),
            ("Lowest", "min"),
        ],
        value="max",
        description="Highlight:",
        layout=widgets.Layout(width="330px"),
        style={"description_width": "initial"},
    )

    show_summary = widgets.Checkbox(
        value=False,
        description="Show summary",
        indent=False,
    )

    summary_title = widgets.HTML("<h4>Summary over aligned slices</h4>")
    summary_out = widgets.Output()

    table_title = widgets.HTML("<h4>Slice-by-slice comparison</h4>")
    table_out = widgets.Output()

    # Put summary title and output in one container so both can be hidden
    summary_box = widgets.VBox([
        summary_title,
        summary_out,
    ])

    # ============================================================
    # Build aligned comparison table
    # ============================================================

    def create_comparison_table():
        selected_subjects = list(subject_select.value)
        selected_methods = list(method_select.value)
        selected_metric = metric_dropdown.value

        filtered_df = df[
            df[subject_col].astype(str).isin(selected_subjects)
            & df["comparison_method"].isin(selected_methods)
        ].copy()

        comparison_df = filtered_df.pivot_table(
            index=slice_identifier_cols,
            columns="comparison_method",
            values=selected_metric,
            aggfunc="mean",
            dropna=False,
        )

        method_columns = [
            method
            for method in selected_methods
            if method in comparison_df.columns
        ]

        comparison_df = comparison_df.reindex(
            columns=method_columns
        )

        # Keep only rows with valid values for all selected methods
        comparison_df = comparison_df.dropna(
            subset=method_columns,
            how="any",
        )

        comparison_df = comparison_df.reset_index()

        return comparison_df, method_columns

    # ============================================================
    # Update display
    # ============================================================

    def update_display(change=None):
        with summary_out:
            clear_output(wait=True)

        with table_out:
            clear_output(wait=True)

        selected_methods = list(method_select.value)
        selected_metric = metric_dropdown.value

        if not subject_select.value:
            with table_out:
                print("Select at least one subject.")
            return

        if not selected_methods:
            with table_out:
                print("Select at least one visible comparison column.")
            return

        comparison_df, method_columns = create_comparison_table()

        if comparison_df.empty:
            with table_out:
                print(
                    f"No aligned subject-slice pairs contain a valid "
                    f"'{selected_metric}' value for every selected method."
                )
            return

        rounded_df = comparison_df.round(decimals)

        # ========================================================
        # Summary
        # ========================================================

        if show_summary.value:
            summary_rows = []

            for method in method_columns:
                values = comparison_df[method]

                summary_rows.append({
                    "method": method,
                    "paired_slice_count": values.notna().sum(),
                    "mean": values.mean(),
                    "std": values.std(),
                    "median": values.median(),
                    "min": values.min(),
                    "max": values.max(),
                })

            summary_df = pd.DataFrame(summary_rows).round(decimals)

            with summary_out:
                n_subjects = comparison_df[subject_col].nunique()
                n_slices = len(comparison_df)

                print(f"Metric: {selected_metric}")
                print(
                    f"Comparing {n_slices:,} aligned subject-slice pairs "
                    f"from {n_subjects} subject(s)."
                )

                display(summary_df)

        # ========================================================
        # Styled slice table
        # ========================================================

        with table_out:

            def highlight_selected_extreme(row):
                styles = [""] * len(row)

                values = row[method_columns]

                if not values.notna().any():
                    return styles

                if highlight_mode.value == "max":
                    target_value = values.max()
                else:
                    target_value = values.min()

                for column in method_columns:
                    if (
                        pd.notna(row[column])
                        and np.isclose(
                            row[column],
                            target_value,
                            equal_nan=False,
                        )
                    ):
                        styles[row.index.get_loc(column)] = (
                            "font-weight: bold; "
                            "background-color: #d9f2d9"
                        )

                return styles

            styled_df = (
                rounded_df.style
                .apply(
                    highlight_selected_extreme,
                    axis=1,
                )
                .format(precision=decimals)
            )

            display(styled_df)

    # ============================================================
    # Toggle summary visibility
    # ============================================================

    def update_summary_visibility(change=None):
        summary_box.layout.display = (
            "flex" if show_summary.value else "none"
        )

        update_display()

    # ============================================================
    # Observe widgets
    # ============================================================

    subject_select.observe(update_display, names="value")
    method_select.observe(update_display, names="value")
    metric_dropdown.observe(update_display, names="value")
    highlight_mode.observe(update_display, names="value")
    show_summary.observe(update_summary_visibility, names="value")

    # ============================================================
    # Display interface
    # ============================================================

    controls = widgets.VBox([
        widgets.HBox([
            subject_select,
            widgets.VBox([
                method_select,
                widgets.HBox([
                    show_all_columns_button,
                    fusion_only_button,
                    column_selection_hint,
                ]),
            ]),
        ]),
        widgets.HBox([
            metric_dropdown,
            highlight_mode,
            show_summary,
        ]),
    ])

    display(
        widgets.VBox([
            controls,
            summary_box,
            table_title,
            table_out,
        ])
    )

    update_summary_visibility()

    return df

In [27]:
all_slice_results = compare_weightings_same_slices(
    path=r"C:\Users\20202310\Desktop\MSc scriptie\MSc_Graduation_Project\fusion_evaluation\Against_GT\Slice results",
)

In [14]:
# ============================================================
# INTERACTIVE PLOTLY BOXPLOTS OF SLICE-BASED METRICS
# ============================================================

from pathlib import Path
import re

import numpy as np
import pandas as pd
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display, clear_output
from pandas.errors import EmptyDataError


def interactive_slice_metric_boxplots_plotly(
    path,
    metric="SurfaceDice@1.0mm",
    recursive=False,
):

    path = Path(path)

    # ============================================================
    # LOAD CSV FILES
    # ============================================================

    if not path.exists():
        raise FileNotFoundError(f"Path does not exist:\n{path}")

    if path.is_file():
        csv_files = [path]

    elif path.is_dir():
        pattern = "**/*.csv" if recursive else "*.csv"
        csv_files = sorted(path.glob(pattern))

        if not csv_files:
            raise FileNotFoundError(
                f"No CSV files found in:\n{path}"
            )

    else:
        raise ValueError("Input must be a CSV file or folder.")

    dataframes = []

    for csv_file in csv_files:

        try:
            temp_df = pd.read_csv(csv_file)

        except EmptyDataError:
            continue

        if temp_df.empty:
            continue

        temp_df["source_file"] = csv_file.stem
        dataframes.append(temp_df)

    if not dataframes:
        raise ValueError("No readable CSV files found.")

    df = pd.concat(
        dataframes,
        ignore_index=True,
        sort=False,
    )

    # ============================================================
    # DETECT SUBJECT + SLICE COLUMN
    # ============================================================

    subject_candidates = [
        "subject",
        "subject_name",
        "subject name",
        "Subject",
        "SubjectName",
    ]

    slice_candidates = [
        "slice",
        "slice_index",
        "slice_idx",
        "slice_nr",
        "slice_number",
        "z",
        "z_index",
        "image_slice",
        "absolute_slice_index",
    ]

    subject_col = next(
        (c for c in subject_candidates if c in df.columns),
        None,
    )

    slice_col = next(
        (c for c in slice_candidates if c in df.columns),
        None,
    )

    if subject_col is None:
        raise ValueError("Could not detect subject column.")

    if slice_col is None:
        raise ValueError("Could not detect slice column.")

    # ============================================================
    # DETERMINE METHOD NAMES
    # ============================================================

    def prettify_method_name(value, column_name=None):

        text = str(value).strip()

        compact = re.sub(
            r"[^a-z0-9]+",
            "",
            text.lower(),
        )

        if "nnunet" in compact:
            return "nnUNet"

        observer_match = re.search(
            r"(?:observer|obs)[_\-\s]*([bcde])",
            text,
            flags=re.IGNORECASE,
        )

        if observer_match:
            return f"Observer {observer_match.group(1).upper()}"

        if (
            column_name is not None
            and "observer" in column_name.lower()
            and compact in {"b", "c", "d", "e"}
        ):
            return f"Observer {compact.upper()}"

        text = re.sub(
            r"^(slice[_\-\s]*results?|results?[_\-\s]*slice)[_\-\s]*",
            "",
            text,
            flags=re.IGNORECASE,
        )

        return text.replace("_", " ").strip()


    explicit_method_columns = [
        c
        for c in [
            "comparison_method",
            "result_name",
            "prediction_name",
            "model",
            "observer",
            "Observer",
        ]
        if c in df.columns
    ]


    def determine_comparison_method(row):

        # Fusion weighting
        if (
            "StapleWeight" in df.columns
            and "BBoxWeight" in df.columns
            and pd.notna(row.get("StapleWeight"))
            and pd.notna(row.get("BBoxWeight"))
        ):

            return (
                f"Staple {row['StapleWeight']:g} | "
                f"BBox {row['BBoxWeight']:g}"
            )

        # Existing method column
        for column in explicit_method_columns:

            value = row.get(column)

            if pd.notna(value) and str(value).strip():
                return prettify_method_name(
                    value,
                    column,
                )

        # Otherwise infer from filename
        return prettify_method_name(
            row["source_file"]
        )


    df["comparison_method"] = df.apply(
        determine_comparison_method,
        axis=1,
    )

    # ============================================================
    # KEEP SLICE-BASED RESULTS
    # ============================================================

    df = df.dropna(
        subset=[
            subject_col,
            slice_col,
            "comparison_method",
        ]
    ).copy()

    df[subject_col] = df[subject_col].astype(str)

    # ============================================================
    # METRIC OPTIONS
    # ============================================================

    excluded_numeric_columns = {
        slice_col,
        "DenseWeight",
        "StapleWeight",
        "BBoxWeight",
        "Threshold",
        "LogitThreshold",
    }

    metric_options = [
        column
        for column in df.select_dtypes(include=np.number).columns
        if column not in excluded_numeric_columns
    ]

    if not metric_options:
        raise ValueError("No numeric metric columns detected.")

    if metric not in metric_options:
        metric = metric_options[0]

    # ============================================================
    # METHOD ORDER
    # ============================================================

    def method_sort_key(method):

        text = str(method)

        # Fusion first
        if text.startswith("Staple "):

            match = re.search(
                r"Staple\s+([0-9.]+)\s*\|\s*BBox\s+([0-9.]+)",
                text,
            )

            if match:
                staple = float(match.group(1))
                bbox = float(match.group(2))

                return (0, staple, bbox)

            return (0, 999, 999)

        # Observers
        observer_match = re.fullmatch(
            r"Observer\s+([B-E])",
            text,
            flags=re.IGNORECASE,
        )

        if observer_match:
            observer_order = "BCDE".index(
                observer_match.group(1).upper()
            )

            return (1, observer_order, 0)

        # nnUNet
        if text.lower() == "nnunet":
            return (2, 0, 0)

        return (3, 0, text)


    method_options = sorted(
        df["comparison_method"]
        .dropna()
        .unique(),
        key=method_sort_key,
    )

    subjects = sorted(
        df[subject_col]
        .dropna()
        .unique()
    )

    # ============================================================
    # WIDGETS
    # ============================================================

    metric_dropdown = widgets.Dropdown(
        options=metric_options,
        value=metric,
        description="Metric:",
        layout=widgets.Layout(width="500px"),
        style={"description_width": "initial"},
    )

    subject_select = widgets.SelectMultiple(
        options=subjects,
        value=tuple(subjects),
        description="Subjects:",
        rows=min(12, max(5, len(subjects))),
        layout=widgets.Layout(width="470px"),
        style={"description_width": "initial"},
    )

    method_select = widgets.SelectMultiple(
        options=method_options,
        value=tuple(method_options),
        description="Methods:",
        rows=min(14, max(5, len(method_options))),
        layout=widgets.Layout(width="470px"),
        style={"description_width": "initial"},
    )

    # ============================================================
    # BUTTONS
    # ============================================================

    select_all_subjects_button = widgets.Button(
        description="All subjects",
        layout=widgets.Layout(width="120px"),
    )

    clear_subjects_button = widgets.Button(
        description="Clear subjects",
        layout=widgets.Layout(width="120px"),
    )

    select_all_methods_button = widgets.Button(
        description="All methods",
        layout=widgets.Layout(width="120px"),
    )

    clear_methods_button = widgets.Button(
        description="Clear methods",
        layout=widgets.Layout(width="120px"),
    )

    output = widgets.Output()

    # ============================================================
    # BUTTON FUNCTIONS
    # ============================================================

    def select_all_subjects(_):
        subject_select.value = tuple(subjects)

    def clear_subjects(_):
        subject_select.value = ()

    def select_all_methods(_):
        method_select.value = tuple(method_options)

    def clear_methods(_):
        method_select.value = ()

    select_all_subjects_button.on_click(
        select_all_subjects
    )

    clear_subjects_button.on_click(
        clear_subjects
    )

    select_all_methods_button.on_click(
        select_all_methods
    )

    clear_methods_button.on_click(
        clear_methods
    )

    # ============================================================
    # UPDATE PLOT
    # ============================================================

    def update_plot(change=None):

        with output:

            clear_output(wait=True)

            selected_metric = metric_dropdown.value
            selected_subjects = list(subject_select.value)
            selected_methods = list(method_select.value)

            if not selected_subjects:
                print("Select at least one subject.")
                return

            if not selected_methods:
                print("Select at least one method.")
                return

            plot_df = df[
                df[subject_col].isin(selected_subjects)
                & df["comparison_method"].isin(selected_methods)
            ].copy()

            plot_df = plot_df.dropna(
                subset=[selected_metric]
            )

            if plot_df.empty:
                print(
                    "No data available for this selection."
                )
                return

            # Preserve selected method order
            available_methods = [
                method
                for method in method_options
                if method in selected_methods
                and method in plot_df[
                    "comparison_method"
                ].unique()
            ]

            # ====================================================
            # PLOT
            # ====================================================

            fig = px.box(
                plot_df,
                x="comparison_method",
                y=selected_metric,

                points="all",

                category_orders={
                    "comparison_method": available_methods
                },

                hover_data={
                    subject_col: True,
                    slice_col: True,
                    "comparison_method": False,
                },

                labels={
                    "comparison_method": "Method",
                    selected_metric: selected_metric,
                },

                title=(
                    f"{selected_metric} per slice "
                    f"— {len(selected_subjects)} subject"
                    f"{'s' if len(selected_subjects) != 1 else ''}"
                ),
            )

            # Make individual slice points slightly compact
            fig.update_traces(
                jitter=0.25,
                pointpos=0,
                marker=dict(
                    size=4,
                    opacity=0.5,
                ),

                # Wider box relative to category width
                width=0.65,
            )

            # ----------------------------------------------------
            # IMPORTANT:
            # Smaller figure width = methods visually closer
            # ----------------------------------------------------

            plot_width = max(
                650,
                95 * len(available_methods),
            )

            fig.update_layout(
                height=600,
                width=plot_width,

                xaxis_title="Method",
                yaxis_title=selected_metric,

                showlegend=False,

                margin=dict(
                    l=70,
                    r=30,
                    t=70,
                    b=120,
                ),
            )

            fig.update_xaxes(
                tickangle=45
            )

            fig.show()

    # ============================================================
    # OBSERVE
    # ============================================================

    metric_dropdown.observe(
        update_plot,
        names="value",
    )

    subject_select.observe(
        update_plot,
        names="value",
    )

    method_select.observe(
        update_plot,
        names="value",
    )

    # ============================================================
    # DISPLAY
    # ============================================================

    controls = widgets.VBox([

        metric_dropdown,

        widgets.HBox([

            widgets.VBox([
                subject_select,

                widgets.HBox([
                    select_all_subjects_button,
                    clear_subjects_button,
                ]),
            ]),

            widgets.VBox([
                method_select,

                widgets.HBox([
                    select_all_methods_button,
                    clear_methods_button,
                ]),
            ]),

        ]),

    ])

    display(
        widgets.VBox([
            controls,
            output,
        ])
    )

    update_plot()

    return df

In [ ]:
boxplot_df = interactive_slice_metric_boxplots_plotly(
    r"C:\Users\20202310\Desktop\MSc scriptie\MSc_Graduation_Project\fusion_evaluation\Against_GT\Slice results"
)

In [30]:
filtered_slice_results = all_slice_results[
    ~all_slice_results["comparison_method"].isin(["nnUNet", "Observer B", "Observer C", "Observer D", "Observer E"])
].copy()

# Drop rows with missing Surface Dice
filtered_slice_results = filtered_slice_results.dropna(
    subset=["SurfaceDice@1.0mm"]
)

# Keep only the best method for each subject-slice
best_idx = filtered_slice_results.groupby(
    ["subject", "slice_idx"]
)["SurfaceDice@1.0mm"].idxmax()

best_slice_results = (
    filtered_slice_results.loc[best_idx]
    .sort_values(["subject", "slice_idx"])
    .reset_index(drop=True)
)

In [31]:
# ============================================================
# Interactive subject selection
# ============================================================

subjects = sorted(best_slice_results["subject"].dropna().unique())

subject_selector = widgets.SelectMultiple(
    options=subjects,
    value=tuple(subjects),
    description="Subjects:",
    layout=widgets.Layout(width="500px", height="250px"),
    style={"description_width": "initial"},
)

select_all_button = widgets.Button(
    description="Select all",
    button_style="",
)

clear_button = widgets.Button(
    description="Clear selection",
    button_style="",
)

output = widgets.Output()


def update_plot(*args):
    selected_subjects = list(subject_selector.value)

    with output:
        output.clear_output(wait=True)

        if not selected_subjects:
            print("Select at least one subject.")
            return

        plot_data = best_slice_results[
            best_slice_results["subject"].isin(selected_subjects)
        ]

        fig, ax = plt.subplots(figsize=(10, 6))

        cmap = plt.get_cmap("tab20")

        for i, subject in enumerate(selected_subjects):
            subject_data = plot_data[plot_data["subject"] == subject]

            ax.scatter(
                subject_data["relative slice idx"],
                subject_data["StapleWeight"],
                label=subject,
                color=cmap(i % cmap.N),
                alpha=0.75,
                s=40,
            )

        ax.set_xlabel("Relative slice index")
        ax.set_ylabel("StapleWeight")
        ax.set_title(
            f"Best Surface Dice weighting per subject-slice ({len(selected_subjects)} subjects)"
        )

        ax.grid(True, alpha=0.3)

        ax.legend(
            title="Subject",
            bbox_to_anchor=(1.02, 1),
            loc="upper left",
            fontsize=8,
        )

        plt.tight_layout()
        plt.show()


def select_all(_):
    subject_selector.value = tuple(subjects)


def clear_selection(_):
    subject_selector.value = ()


subject_selector.observe(update_plot, names="value")
select_all_button.on_click(select_all)
clear_button.on_click(clear_selection)

display(
    widgets.VBox(
        [
            subject_selector,
            widgets.HBox([select_all_button, clear_button]),
            output,
        ]
    )
)

update_plot()

In [32]:
best_slice_results

,subject,slice_idx,HD_mm,HD95_mm,MSD_mm,ASSD_mm,SurfaceDice@1.0mm,CentroidDistance_mm,PredictionPixels,GroundTruthPixels,...,SurfaceDice@2.0mm,PredictionArea_mm2,GroundTruthArea_mm2,AbsAreaDifference_mm2,RelativeAreaDifference_percent,source_file,StapleWeight,BBoxWeight,LogitThreshold,comparison_method
0,newAcq_050f229dc2bdb64c,29,5.664532,4.680933,2.283067,2.016018,0.287879,2.387574,1650,1242,...,0.606061,362.626189,272.958622,89.667567,32.850242,slice_results_Staple_0.8_bbox_0.2,0.8,0.2,0.0,Staple 0.8 | BBox 0.2
1,newAcq_050f229dc2bdb64c,30,5.049131,3.314917,0.990438,1.063389,0.613272,1.019402,3811,3954,...,0.910755,837.556609,868.984213,31.427603,-3.616591,slice_results_Staple_0.5_bbox_0.5,0.5,0.5,0.0,Staple 0.5 | BBox 0.5
2,newAcq_050f229dc2bdb64c,31,3.977900,3.023238,1.216980,1.254868,0.508032,1.741503,5007,4847,...,0.805221,1100.405653,1065.241901,35.163752,3.301011,slice_results_Staple_0.1_bbox_0.9,0.1,0.9,0.0,Staple 0.1 | BBox 0.9
3,newAcq_050f229dc2bdb64c,32,8.200651,6.307059,1.598165,1.903664,0.489443,1.687929,5147,6317,...,0.650672,1131.173936,1388.308870,257.134934,-18.521450,slice_results_Staple_0.3_bbox_0.7,0.3,0.7,0.0,Staple 0.3 | BBox 0.7
4,newAcq_050f229dc2bdb64c,33,5.049131,3.750400,1.751540,1.828495,0.301226,1.284922,6505,7589,...,0.572680,1429.626278,1667.860695,238.234417,-14.283832,slice_results_Staple_0.6_bbox_0.4,0.6,0.4,0.0,Staple 0.6 | BBox 0.4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
185,newAcq_4a136e8fe320bd13,46,7.955800,6.921749,2.565934,2.759426,0.155050,0.191547,8376,10557,...,0.463727,1840.822399,2320.148288,479.325890,-20.659278,slice_results_Staple_0.0_bbox_1.0,0.0,1.0,0.0,Staple 0 | BBox 1
186,newAcq_4a136e8fe320bd13,47,9.492477,7.964046,2.665975,2.862829,0.394081,1.407819,7230,9308,...,0.479751,1588.962027,2045.651252,456.689224,-22.324882,slice_results_Staple_0.1_bbox_0.9,0.1,0.9,0.0,Staple 0.1 | BBox 0.9
187,newAcq_4a136e8fe320bd13,48,10.032758,8.160783,2.439111,2.743722,0.262397,1.566465,4862,5464,...,0.526860,1068.538503,1200.842119,132.303616,-11.017570,slice_results_Staple_1.0_bbox_0.0,1.0,0.0,0.0,Staple 1 | BBox 0
188,newAcq_4a136e8fe320bd13,49,5.467101,4.894416,2.422958,2.261043,0.328947,2.761000,3590,2782,...,0.489474,788.986678,611.409732,177.576946,29.043853,slice_results_Staple_0.1_bbox_0.9,0.1,0.9,0.0,Staple 0.1 | BBox 0.9


In [36]:
# ============================================================
# INTERACTIVE DESCRIPTIVE STATISTICS
# CLINICAL BOUNDARY METRIC
# ============================================================

import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output


# ------------------------------------------------------------
# LOAD DATA
# ------------------------------------------------------------

volume_df = pd.read_csv(
    "fusion_evaluation/clinical_boundary_volume_results.csv"
)

slice_df = pd.read_csv(
    "fusion_evaluation/clinical_boundary_slice_results.csv"
)

metric = "ContourWithinClinicalBoundary"

# Convert fraction -> percentage
volume_df["ClinicalBoundary_percent"] = (
    volume_df[metric] * 100
)

slice_df["ClinicalBoundary_percent"] = (
    slice_df[metric] * 100
)

plot_metric = "ClinicalBoundary_percent"


# ------------------------------------------------------------
# CREATE WEIGHTING LABELS
# ------------------------------------------------------------

def add_weight_labels(df):

    df = df.copy()

    df["Weighting"] = (
        "Staple "
        + df["StapleWeight"].round(1).astype(str)
        + " | BBox "
        + df["BBoxWeight"].round(1).astype(str)
    )

    return df


volume_df = add_weight_labels(volume_df)
slice_df = add_weight_labels(slice_df)


weight_table = (
    volume_df[
        ["Weighting", "StapleWeight", "BBoxWeight"]
    ]
    .drop_duplicates()
    .sort_values("StapleWeight")
)

weighting_options = weight_table["Weighting"].tolist()

subject_options = sorted(
    slice_df["subject"].unique()
)


# ============================================================
# TABLE FUNCTION
# ============================================================

def compute_clinical_boundary_statistics(
    analysis_level,
    selected_weightings,
    selected_subjects,
    round_decimals
):

    selected_weightings = list(selected_weightings)
    selected_subjects = list(selected_subjects)

    if len(selected_weightings) == 0:
        print("Select at least one weighting.")
        return

    # --------------------------------------------------------
    # SELECT DATASET
    # --------------------------------------------------------

    if analysis_level == "Volume":

        df = volume_df[
            volume_df["Weighting"].isin(
                selected_weightings
            )
        ].copy()

    else:

        if len(selected_subjects) == 0:
            print("Select at least one subject.")
            return

        df = slice_df[
            slice_df["Weighting"].isin(
                selected_weightings
            )
            & slice_df["subject"].isin(
                selected_subjects
            )
        ].copy()

    # --------------------------------------------------------
    # DESCRIPTIVE STATISTICS
    # --------------------------------------------------------

    stats = (
        df
        .groupby("Weighting")[plot_metric]
        .describe()
    )

    stats["median"] = (
        df
        .groupby("Weighting")[plot_metric]
        .median()
    )

    stats["iqr"] = (
        stats["75%"]
        - stats["25%"]
    )

    stats = stats.reset_index()

    # Preserve numerical weighting order
    stats = stats.merge(
        weight_table,
        on="Weighting",
        how="left"
    )

    stats = stats.sort_values(
        "StapleWeight"
    )

    stats = stats[
        [
            "Weighting",
            "count",
            "mean",
            "std",
            "min",
            "25%",
            "median",
            "75%",
            "iqr",
            "max",
        ]
    ]

    print(
        f"{analysis_level}-wise clinical boundary statistics"
    )

    if analysis_level == "Volume":
        print(
            f"N subjects: "
            f"{df['subject'].nunique()}"
        )
    else:
        print(
            f"N subjects: {df['subject'].nunique()} | "
            f"N slices: {len(df)}"
        )

    display(
        stats.round(round_decimals)
    )


# ============================================================
# WIDGETS
# ============================================================

analysis_selector = widgets.ToggleButtons(
    options=["Volume", "Slice"],
    value="Volume",
    description="Analysis:"
)

weighting_selector = widgets.SelectMultiple(
    options=weighting_options,
    value=tuple(weighting_options),
    description="Weightings:",
    rows=min(11, len(weighting_options)),
    layout=widgets.Layout(
        width="420px"
    )
)

subject_selector = widgets.SelectMultiple(
    options=subject_options,
    value=tuple(subject_options),
    description="Subjects:",
    rows=min(10, len(subject_options)),
    layout=widgets.Layout(
        width="420px"
    )
)

round_slider = widgets.IntSlider(
    value=3,
    min=0,
    max=6,
    step=1,
    description="Decimals:"
)


# ============================================================
# BUTTONS
# ============================================================

all_weights_button = widgets.Button(
    description="All weightings"
)

clear_weights_button = widgets.Button(
    description="Clear weightings"
)

all_subjects_button = widgets.Button(
    description="All subjects"
)

clear_subjects_button = widgets.Button(
    description="Clear subjects"
)


def select_all_weights(_):
    weighting_selector.value = tuple(
        weighting_options
    )


def clear_weights(_):
    weighting_selector.value = ()


def select_all_subjects(_):
    subject_selector.value = tuple(
        subject_options
    )


def clear_subjects(_):
    subject_selector.value = ()


all_weights_button.on_click(
    select_all_weights
)

clear_weights_button.on_click(
    clear_weights
)

all_subjects_button.on_click(
    select_all_subjects
)

clear_subjects_button.on_click(
    clear_subjects
)


# ============================================================
# INTERACTIVE OUTPUT
# ============================================================

out = widgets.interactive_output(
    compute_clinical_boundary_statistics,
    {
        "analysis_level": analysis_selector,
        "selected_weightings": weighting_selector,
        "selected_subjects": subject_selector,
        "round_decimals": round_slider,
    }
)


display(
    widgets.VBox([
        analysis_selector,

        widgets.HBox([
            widgets.VBox([
                weighting_selector,
                widgets.HBox([
                    all_weights_button,
                    clear_weights_button
                ])
            ]),

            widgets.VBox([
                subject_selector,
                widgets.HBox([
                    all_subjects_button,
                    clear_subjects_button
                ])
            ])
        ]),

        round_slider
    ]),
    out
)

Output()

In [37]:
# ============================================================
# INTERACTIVE CLINICAL BOUNDARY PLOTS
# ============================================================

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output


# ============================================================
# LOAD DATA
# ============================================================

volume_df = pd.read_csv(
    "fusion_evaluation/clinical_boundary_volume_results.csv"
)

slice_df = pd.read_csv(
    "fusion_evaluation/clinical_boundary_slice_results.csv"
)

metric = "ContourWithinClinicalBoundary"

# Convert fraction -> percentage
volume_df["ClinicalBoundary_percent"] = (
    volume_df[metric] * 100
)

slice_df["ClinicalBoundary_percent"] = (
    slice_df[metric] * 100
)

plot_metric = "ClinicalBoundary_percent"


# ============================================================
# CREATE WEIGHTING LABELS
# ============================================================

def make_weight_label(df):
    return (
        "Staple "
        + df["StapleWeight"].round(1).astype(str)
        + " | BBox "
        + df["BBoxWeight"].round(1).astype(str)
    )


volume_df["Weighting"] = make_weight_label(volume_df)
slice_df["Weighting"] = make_weight_label(slice_df)

# Keep weightings numerically ordered
weight_table = (
    volume_df[
        ["Weighting", "StapleWeight", "BBoxWeight"]
    ]
    .drop_duplicates()
    .sort_values("StapleWeight")
)

weightings = weight_table["Weighting"].tolist()

subjects = sorted(
    slice_df["subject"].unique()
)


# ============================================================
# WIDGETS
# ============================================================

weight_selector = widgets.SelectMultiple(
    options=weightings,
    value=tuple(weightings),
    description="Weightings:",
    layout=widgets.Layout(
        width="420px",
        height="210px"
    )
)

subject_selector = widgets.SelectMultiple(
    options=subjects,
    value=tuple(subjects),
    description="Subjects:",
    layout=widgets.Layout(
        width="420px",
        height="210px"
    )
)

show_volume_points = widgets.Checkbox(
    value=True,
    description="Show individual subjects"
)

show_paired_lines = widgets.Checkbox(
    value=False,
    description="Show paired subject lines"
)

slice_x_mode = widgets.Dropdown(
    options=[
        ("Absolute slice index", "absolute"),
        ("Relative slice position", "relative"),
    ],
    value="relative",
    description="Slice x-axis:",
)

select_all_weights_button = widgets.Button(
    description="All weightings"
)

clear_weights_button = widgets.Button(
    description="Clear weightings"
)

select_all_subjects_button = widgets.Button(
    description="All subjects"
)

clear_subjects_button = widgets.Button(
    description="Clear subjects"
)

output_volume = widgets.Output()
output_slice = widgets.Output()


# ============================================================
# BUTTON FUNCTIONS
# ============================================================

def select_all_weights(_):
    weight_selector.value = tuple(weightings)


def clear_weights(_):
    weight_selector.value = ()


def select_all_subjects(_):
    subject_selector.value = tuple(subjects)


def clear_subjects(_):
    subject_selector.value = ()


select_all_weights_button.on_click(select_all_weights)
clear_weights_button.on_click(clear_weights)

select_all_subjects_button.on_click(select_all_subjects)
clear_subjects_button.on_click(clear_subjects)


# ============================================================
# VOLUME PLOT
# ============================================================

def update_volume_plot(*args):

    selected_weights = list(weight_selector.value)

    with output_volume:

        clear_output(wait=True)

        if len(selected_weights) == 0:
            print("Select at least one weighting.")
            return

        df = volume_df[
            volume_df["Weighting"].isin(selected_weights)
        ].copy()

        fig = go.Figure()

        # ----------------------------------------------------
        # Boxplots
        # ----------------------------------------------------

        for weighting in selected_weights:

            sub = df[
                df["Weighting"] == weighting
            ]

            fig.add_trace(
                go.Box(
                    x=[weighting] * len(sub),
                    y=sub[plot_metric],
                    name=weighting,
                    boxpoints=(
                        "all"
                        if show_volume_points.value
                        else False
                    ),
                    jitter=0.25,
                    pointpos=0,
                    showlegend=False,
                )
            )

        # ----------------------------------------------------
        # Paired subject lines
        # ----------------------------------------------------

        if show_paired_lines.value:

            paired = df.pivot(
                index="subject",
                columns="Weighting",
                values=plot_metric
            )

            for subject in paired.index:

                available_weights = [
                    w for w in selected_weights
                    if w in paired.columns
                    and pd.notna(
                        paired.loc[subject, w]
                    )
                ]

                values = [
                    paired.loc[subject, w]
                    for w in available_weights
                ]

                fig.add_trace(
                    go.Scatter(
                        x=available_weights,
                        y=values,
                        mode="lines",
                        line=dict(
                            width=1
                        ),
                        opacity=0.25,
                        showlegend=False,
                        hovertext=[subject] * len(values),
                        hovertemplate=(
                            "%{hovertext}<br>"
                            "%{x}<br>"
                            "%{y:.1f}%"
                            "<extra></extra>"
                        ),
                    )
                )

        fig.update_layout(
            title=(
                "Volume-wise contour within "
                "clinical boundary"
            ),
            xaxis_title="Weighting",
            yaxis_title=(
                "Contour within clinical boundary (%)"
            ),
            yaxis=dict(
                range=[0, 100]
            ),
            template="plotly_white",
            height=550,
            margin=dict(
                l=70,
                r=30,
                t=70,
                b=130
            ),
        )

        fig.update_xaxes(
            tickangle=45
        )

        fig.show()


# ============================================================
# SLICE PLOT
# ============================================================

def update_slice_plot(*args):

    selected_weights = list(weight_selector.value)
    selected_subjects = list(subject_selector.value)

    with output_slice:

        clear_output(wait=True)

        if len(selected_weights) == 0:
            print("Select at least one weighting.")
            return

        if len(selected_subjects) == 0:
            print("Select at least one subject.")
            return

        df = slice_df[
            slice_df["Weighting"].isin(selected_weights)
            & slice_df["subject"].isin(selected_subjects)
        ].copy()

        if slice_x_mode.value == "relative":

            df["relative_slice"] = (
                df.groupby(
                    ["subject", "Weighting"]
                )["slice_idx"]
                .transform(
                    lambda x:
                    (x - x.min())
                    / (x.max() - x.min())
                    if x.max() != x.min()
                    else 0
                )
            )

            x_col = "relative_slice"
            x_label = "Relative slice position"

        else:

            x_col = "slice_idx"
            x_label = "Slice index"

        fig = px.line(
            df,
            x=x_col,
            y=plot_metric,
            color="Weighting",
            line_group="subject",
            markers=True,
            hover_data={
                "subject": True,
                "slice_idx": True,
                "StapleWeight": True,
                "BBoxWeight": True,
                plot_metric: ":.1f",
            },
        )

        fig.update_layout(
            title=(
                "Slice-wise contour within "
                "clinical boundary"
            ),
            xaxis_title=x_label,
            yaxis_title=(
                "Contour within clinical boundary (%)"
            ),
            yaxis=dict(
                range=[0, 100]
            ),
            template="plotly_white",
            height=600,
            legend_title="Weighting",
        )

        fig.show()


# ============================================================
# CONNECT WIDGETS
# ============================================================

weight_selector.observe(
    update_volume_plot,
    names="value"
)

weight_selector.observe(
    update_slice_plot,
    names="value"
)

subject_selector.observe(
    update_slice_plot,
    names="value"
)

show_volume_points.observe(
    update_volume_plot,
    names="value"
)

show_paired_lines.observe(
    update_volume_plot,
    names="value"
)

slice_x_mode.observe(
    update_slice_plot,
    names="value"
)


# ============================================================
# DISPLAY
# ============================================================

controls = widgets.VBox([

    widgets.HTML(
        "<h3>Clinical boundary analysis</h3>"
    ),

    widgets.HBox([
        widgets.VBox([
            weight_selector,
            widgets.HBox([
                select_all_weights_button,
                clear_weights_button,
            ]),
        ]),

        widgets.VBox([
            subject_selector,
            widgets.HBox([
                select_all_subjects_button,
                clear_subjects_button,
            ]),
        ]),
    ]),

    widgets.HBox([
        show_volume_points,
        show_paired_lines,
        slice_x_mode,
    ]),
])

display(controls)

display(
    widgets.HTML(
        "<h4>Volume-wise analysis</h4>"
    )
)
display(output_volume)

display(
    widgets.HTML(
        "<h4>Slice-wise analysis</h4>"
    )
)
display(output_slice)


# Initial plots
update_volume_plot()
update_slice_plot()

HTML(value='<h4>Volume-wise analysis</h4>')

Output()

HTML(value='<h4>Slice-wise analysis</h4>')

Output()

In [14]:
# ============================================================
# INTERACTIVE SUBJECT + LOGIT WEIGHTING + RECONTOUR VIEWER
# ============================================================

from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import ipywidgets as widgets

from IPython.display import display
from skimage.measure import find_contours


def interactive_weighted_logits_recontour_viewer(
    root,
    subjects,
    logits_folder,
    volume_of_interest="CTVT",
    figsize=(8, 8),
    zoom_fraction=0.30,
    initial_weight=0.50,
    logit_threshold=0.0,
    consensus_indices = [0,1,2,3]
):
    """
    Interactive viewer for weighted logit fusion.

    The generated segmentation is calculated as:

        fused_logits =
            nietjes_weight * Dense_and_nietjes
            + (1 - nietjes_weight) * Uncertainty_bboxes

        generated_segmentation = fused_logits > logit_threshold

    Displays:
    - Ground truth: data.gt
    - Original nnUNet mask: data.mask
    - Generated weighted segmentation
    - Consensus recontour: data.consensus_recontour
    - Individual observer recontours: data.observer_recontours

    Parameters
    ----------
    root
        Root folder supplied to DataLoader.

    subjects
        Subject list used by DataLoader.

    logits_folder
        Folder containing files named:

            {subject_name}_logits.npz

        Each file must contain:

            "Dense_and_nietjes"
            "Uncertainty_bboxes"

    volume_of_interest
        Volume of interest supplied to DataLoader.

    figsize
        Matplotlib figure size.

    zoom_fraction
        Fraction of the original image shown when zoom is enabled.

    initial_weight
        Initial weight assigned to Dense_and_nietjes.
        The bbox weight is automatically 1 - initial_weight.

    logit_threshold
        Threshold applied to the fused logits.
        For raw logits, 0.0 is equivalent to a sigmoid threshold of 0.5.
    """

    root = Path(root)
    logits_folder = Path(logits_folder)
    subjects = list(subjects)

    if len(subjects) == 0:
        raise ValueError("The subjects list is empty.")

    # --------------------------------------------------------
    # Widget definitions
    # --------------------------------------------------------

    subject_dropdown = widgets.Dropdown(
        options=[
            (str(subject), subject_nr)
            for subject_nr, subject in enumerate(subjects)
        ],
        value=0,
        description="Subject:",
        style={"description_width": "initial"},
        layout=widgets.Layout(width="500px"),
    )

    slice_slider = widgets.IntSlider(
        min=0,
        max=1,
        value=0,
        step=1,
        description="Slice:",
        continuous_update=False,
        style={"description_width": "initial"},
        layout=widgets.Layout(width="600px"),
    )

    weight_slider = widgets.FloatSlider(
        min=0.0,
        max=1.0,
        value=float(initial_weight),
        step=0.05,
        description="Nietjes/staples weight:",
        readout_format=".2f",
        continuous_update=False,
        style={"description_width": "initial"},
        layout=widgets.Layout(width="600px"),
    )

    zoom_checkbox = widgets.Checkbox(
        value=False,
        description="Zoom",
        indent=False,
    )

    show_gt_checkbox = widgets.Checkbox(
        value=True,
        description="Ground truth",
        indent=False,
    )

    show_original_checkbox = widgets.Checkbox(
        value=True,
        description="Original mask",
        indent=False,
    )

    show_generated_checkbox = widgets.Checkbox(
        value=True,
        description="Weighted segmentation",
        indent=False,
    )

    show_consensus_checkbox = widgets.Checkbox(
        value=True,
        description="Consensus recontour",
        indent=False,
    )

    observer_widget_box = widgets.VBox()

    output = widgets.Output()

    subject_cache = {}
    current_data = {
        "subject_nr": None,
        "subject_name": None,
        "img": None,
        "gt": None,
        "original_mask": None,
        "dense_nietjes_logits": None,
        "bbox_logits": None,
        "observer_recontours": [],
        "observer_names": [],
        "consensus": None,
        "observer_checkboxes": [],
    }

    # --------------------------------------------------------
    # Helper functions
    # --------------------------------------------------------

    def prepare_volume(array, name):
        """
        Convert an array to a 3D volume with shape (z, y, x).

        Singleton dimensions are removed automatically, for example:
            (1, z, y, x) -> (z, y, x)
            (z, 1, y, x) -> (z, y, x)
        """

        array = np.asarray(array)
        array = np.squeeze(array)

        if array.ndim != 3:
            raise ValueError(
                f"{name} must become a 3D array after removing singleton "
                f"dimensions. Got shape {array.shape}."
            )

        return array

    def normalize_observer_recontours(observer_recontours):
        """
        Support observer recontours stored as either:

        1. Dictionary:
               {"B": mask_B, "C": mask_C, ...}

        2. List or tuple:
               [mask_B, mask_C, mask_D, mask_E]

        3. Single 4D NumPy array:
               (n_observers, z, y, x)
        """

        if observer_recontours is None:
            return [], []

        if isinstance(observer_recontours, dict):
            names = list(observer_recontours.keys())
            masks = [
                prepare_volume(
                    observer_recontours[name],
                    f"Observer {name}",
                ).astype(bool)
                for name in names
            ]

            return names, masks

        if isinstance(observer_recontours, np.ndarray):
            observer_recontours = np.asarray(observer_recontours)

            if observer_recontours.ndim == 4:
                masks = [
                    prepare_volume(
                        observer_recontours[i],
                        f"Observer {i + 1}",
                    ).astype(bool)
                    for i in range(observer_recontours.shape[0])
                ]
            else:
                masks = [
                    prepare_volume(
                        observer_recontours,
                        "Observer recontour",
                    ).astype(bool)
                ]

        else:
            masks = [
                prepare_volume(
                    mask,
                    f"Observer {i + 1}",
                ).astype(bool)
                for i, mask in enumerate(observer_recontours)
            ]

        default_names = ["B", "C", "D", "E"]

        if len(masks) <= len(default_names):
            names = default_names[:len(masks)]
        else:
            names = [
                str(i + 1)
                for i in range(len(masks))
            ]

        return names, masks

    def draw_contour(
        ax,
        mask,
        color,
        linewidth=1.8,
        linestyle="-",
    ):
        """Draw all contours contained in one binary 2D mask."""

        if mask is None or not np.any(mask):
            return

        contours = find_contours(
            mask.astype(float),
            level=0.5,
        )

        for contour in contours:
            ax.plot(
                contour[:, 1],
                contour[:, 0],
                color=color,
                linewidth=linewidth,
                linestyle=linestyle,
            )

    def validate_shape(reference, array, name):
        if array is not None and array.shape != reference.shape:
            raise ValueError(
                f"{name} has shape {array.shape}, but the image has "
                f"shape {reference.shape}."
            )

    def load_subject(subject_nr):
        """
        Load the subject, recontours, consensus and saved logits.

        Loaded subjects are cached so switching back to an earlier subject
        does not reload everything from disk.
        """

        if subject_nr in subject_cache:
            return subject_cache[subject_nr]

        data = DataLoader(
            parentfolder=root,
            subject_nr=subject_nr,
            volume_of_interest=volume_of_interest,
            verbose=True,
        )

        data.load_recontours()
        data.load_consensus(indices = consensus_indices)

        subject_name = data.subject_name

        logits_path = (
            logits_folder /
            f"{subject_name}_logits.npz"
        )

        if not logits_path.exists():
            raise FileNotFoundError(
                f"No saved logits were found for {subject_name}:\n"
                f"{logits_path}"
            )

        with np.load(logits_path) as saved_logits:
            required_keys = {
                "Dense_and_nietjes",
                "Uncertainty_bboxes",
            }

            missing_keys = (
                required_keys -
                set(saved_logits.files)
            )

            if missing_keys:
                raise KeyError(
                    f"The following arrays are missing from "
                    f"{logits_path.name}: {sorted(missing_keys)}\n\n"
                    f"Available arrays: {saved_logits.files}"
                )

            dense_nietjes_logits = prepare_volume(
                saved_logits["Dense_and_nietjes"],
                "Dense_and_nietjes logits",
            ).astype(np.float32)

            bbox_logits = prepare_volume(
                saved_logits["Uncertainty_bboxes"],
                "Uncertainty_bboxes logits",
            ).astype(np.float32)

        img = prepare_volume(
            data.img,
            "Image",
        )

        gt = prepare_volume(
            data.gt,
            "Ground truth",
        ).astype(bool)

        original_mask = prepare_volume(
            data.mask,
            "Original mask",
        ).astype(bool)

        consensus = prepare_volume(
            data.consensus_recontour,
            "Consensus recontour",
        ).astype(bool)

        observer_names, observer_recontours = (
            normalize_observer_recontours(
                data.observer_recontours
            )
        )

        validate_shape(img, gt, "Ground truth")
        validate_shape(img, original_mask, "Original mask")
        validate_shape(img, consensus, "Consensus recontour")

        validate_shape(
            img,
            dense_nietjes_logits,
            "Dense_and_nietjes logits",
        )

        validate_shape(
            img,
            bbox_logits,
            "Uncertainty_bboxes logits",
        )

        for observer_name, observer_mask in zip(
            observer_names,
            observer_recontours,
        ):
            validate_shape(
                img,
                observer_mask,
                f"Observer {observer_name}",
            )

        loaded = {
            "subject_nr": subject_nr,
            "subject_name": subject_name,
            "img": img,
            "gt": gt,
            "original_mask": original_mask,
            "dense_nietjes_logits": dense_nietjes_logits,
            "bbox_logits": bbox_logits,
            "observer_recontours": observer_recontours,
            "observer_names": observer_names,
            "consensus": consensus,
        }

        subject_cache[subject_nr] = loaded

        return loaded

    def build_observer_checkboxes():
        """Create one checkbox for each available observer."""

        checkboxes = []

        for observer_name in current_data["observer_names"]:
            checkbox = widgets.Checkbox(
                value=True,
                description=f"Observer {observer_name}",
                indent=False,
            )

            checkbox.observe(
                update_plot,
                names="value",
            )

            checkboxes.append(checkbox)

        current_data["observer_checkboxes"] = checkboxes

        observer_widget_box.children = tuple(checkboxes)

    def determine_crop(
        z,
        generated_segmentation,
    ):
        """
        Determine the zoom crop from all currently visible structures.
        """

        image_slice = current_data["img"][z]
        height, width = image_slice.shape

        if not zoom_checkbox.value:
            return 0, height, 0, width

        foreground = np.zeros(
            (height, width),
            dtype=bool,
        )

        if show_gt_checkbox.value:
            foreground |= current_data["gt"][z]

        if show_original_checkbox.value:
            foreground |= current_data["original_mask"][z]

        if show_generated_checkbox.value:
            foreground |= generated_segmentation[z]

        if show_consensus_checkbox.value:
            foreground |= current_data["consensus"][z]

        for observer_mask, checkbox in zip(
            current_data["observer_recontours"],
            current_data["observer_checkboxes"],
        ):
            if checkbox.value:
                foreground |= observer_mask[z]

        if foreground.any():
            ys, xs = np.where(foreground)

            center_y = int(
                np.round(np.mean(ys))
            )

            center_x = int(
                np.round(np.mean(xs))
            )

        else:
            center_y = height // 2
            center_x = width // 2

        crop_height = max(
            2,
            int(round(height * zoom_fraction)),
        )

        crop_width = max(
            2,
            int(round(width * zoom_fraction)),
        )

        y0 = center_y - crop_height // 2
        x0 = center_x - crop_width // 2

        y0 = max(
            0,
            min(y0, height - crop_height),
        )

        x0 = max(
            0,
            min(x0, width - crop_width),
        )

        y1 = min(
            height,
            y0 + crop_height,
        )

        x1 = min(
            width,
            x0 + crop_width,
        )

        return y0, y1, x0, x1

    # --------------------------------------------------------
    # Plotting
    # --------------------------------------------------------

    def update_plot(change=None):
        if current_data["img"] is None:
            return

        with output:
            output.clear_output(wait=True)

            z = int(slice_slider.value)

            nietjes_weight = float(
                weight_slider.value
            )

            bbox_weight = 1.0 - nietjes_weight

            fused_logits = (
                nietjes_weight
                * current_data["dense_nietjes_logits"]
                +
                bbox_weight
                * current_data["bbox_logits"]
            )

            generated_segmentation = (
                fused_logits > logit_threshold
            )

            y0, y1, x0, x1 = determine_crop(
                z=z,
                generated_segmentation=generated_segmentation,
            )

            image_crop = current_data["img"][
                z,
                y0:y1,
                x0:x1,
            ]

            gt_crop = current_data["gt"][
                z,
                y0:y1,
                x0:x1,
            ]

            original_crop = current_data["original_mask"][
                z,
                y0:y1,
                x0:x1,
            ]

            generated_crop = generated_segmentation[
                z,
                y0:y1,
                x0:x1,
            ]

            consensus_crop = current_data["consensus"][
                z,
                y0:y1,
                x0:x1,
            ]

            observer_crops = [
                observer_mask[
                    z,
                    y0:y1,
                    x0:x1,
                ]
                for observer_mask
                in current_data["observer_recontours"]
            ]

            # ------------------------------------------------
            # Contour colors
            # ------------------------------------------------

            color_gt = "#7fc97f"
            color_original = "#f28e2b"
            color_generated = "#ffd92f"
            color_consensus = "#00ffff"

            observer_colors = [
                "#ffcccc",
                "#ff8a8a",
                "#e63946",
                "#9d0208",
                "#7f0000",
                "#4a0000",
            ]

            fig, ax = plt.subplots(
                figsize=figsize
            )

            ax.imshow(
                image_crop,
                cmap="gray",
            )

            legend_handles = []

            if show_gt_checkbox.value:
                draw_contour(
                    ax=ax,
                    mask=gt_crop,
                    color=color_gt,
                    linewidth=1.8,
                )

                legend_handles.append(
                    mlines.Line2D(
                        [],
                        [],
                        color=color_gt,
                        linewidth=2,
                        label="Ground truth",
                    )
                )

            if show_original_checkbox.value:
                draw_contour(
                    ax=ax,
                    mask=original_crop,
                    color=color_original,
                    linewidth=1.8,
                )

                legend_handles.append(
                    mlines.Line2D(
                        [],
                        [],
                        color=color_original,
                        linewidth=2,
                        label="Original mask",
                    )
                )

            if show_consensus_checkbox.value:
                draw_contour(
                    ax=ax,
                    mask=consensus_crop,
                    color=color_consensus,
                    linewidth=2.2,
                    linestyle="--",
                )

                legend_handles.append(
                    mlines.Line2D(
                        [],
                        [],
                        color=color_consensus,
                        linewidth=2.2,
                        linestyle="--",
                        label="Consensus recontour",
                    )
                )

            for i, (
                observer_name,
                observer_crop,
                observer_checkbox,
            ) in enumerate(
                zip(
                    current_data["observer_names"],
                    observer_crops,
                    current_data["observer_checkboxes"],
                )
            ):
                if not observer_checkbox.value:
                    continue

                observer_color = observer_colors[
                    i % len(observer_colors)
                ]

                draw_contour(
                    ax=ax,
                    mask=observer_crop,
                    color=observer_color,
                    linewidth=1.8,
                )

                legend_handles.append(
                    mlines.Line2D(
                        [],
                        [],
                        color=observer_color,
                        linewidth=2,
                        label=f"Observer {observer_name}",
                    )
                )

            # Draw generated segmentation last so it remains visible.
            if show_generated_checkbox.value:
                draw_contour(
                    ax=ax,
                    mask=generated_crop,
                    color=color_generated,
                    linewidth=2.2,
                )

                legend_handles.append(
                    mlines.Line2D(
                        [],
                        [],
                        color=color_generated,
                        linewidth=2.2,
                        label="Weighted segmentation",
                    )
                )

            ax.set_title(
                f"{current_data['subject_name']} — slice {z}\n"
                f"Nietjes/staples: {nietjes_weight:.2f} | "
                f"Bboxes: {bbox_weight:.2f}"
            )

            ax.set_axis_off()

            if legend_handles:
                ax.legend(
                    handles=legend_handles,
                    loc="upper right",
                    framealpha=0.90,
                    fontsize=9,
                )

            plt.tight_layout()
            plt.show()

    # --------------------------------------------------------
    # Subject switching
    # --------------------------------------------------------

    def update_subject(change=None):
        subject_nr = int(
            subject_dropdown.value
        )

        with output:
            output.clear_output(wait=True)
            print("Loading subject...")

        try:
            loaded = load_subject(
                subject_nr
            )

        except Exception as error:
            with output:
                output.clear_output(wait=True)
                print(
                    f"Could not load subject "
                    f"{subjects[subject_nr]}:\n\n{error}"
                )

            current_data["img"] = None
            return

        current_data.update(loaded)

        n_slices = current_data["img"].shape[0]

        slice_slider.max = n_slices - 1

        foreground = (
            current_data["gt"]
            |
            current_data["original_mask"]
            |
            current_data["consensus"]
        )

        if foreground.any():
            foreground_slices = np.where(
                np.any(
                    foreground,
                    axis=(1, 2),
                )
            )[0]

            initial_slice = int(
                np.round(
                    np.mean(foreground_slices)
                )
            )

        else:
            initial_slice = n_slices // 2

        slice_slider.value = initial_slice

        build_observer_checkboxes()
        update_plot()

    # --------------------------------------------------------
    # Connect widget events
    # --------------------------------------------------------

    subject_dropdown.observe(
        update_subject,
        names="value",
    )

    slice_slider.observe(
        update_plot,
        names="value",
    )

    weight_slider.observe(
        update_plot,
        names="value",
    )

    zoom_checkbox.observe(
        update_plot,
        names="value",
    )

    show_gt_checkbox.observe(
        update_plot,
        names="value",
    )

    show_original_checkbox.observe(
        update_plot,
        names="value",
    )

    show_generated_checkbox.observe(
        update_plot,
        names="value",
    )

    show_consensus_checkbox.observe(
        update_plot,
        names="value",
    )

    # --------------------------------------------------------
    # Display interface
    # --------------------------------------------------------

    display(
        widgets.VBox(
            [
                subject_dropdown,
                slice_slider,
                weight_slider,
                widgets.HTML(
                    value=(
                        "<b>Bbox weight = "
                        "1 − nietjes/staples weight</b>"
                    )
                ),
                widgets.HBox(
                    [
                        zoom_checkbox,
                        show_gt_checkbox,
                        show_original_checkbox,
                    ]
                ),
                widgets.HBox(
                    [
                        show_generated_checkbox,
                        show_consensus_checkbox,
                    ]
                ),
                widgets.HTML(
                    value="<b>Individual recontours</b>"
                ),
                observer_widget_box,
                output,
            ]
        )
    )

    # Load the initially selected subject.
    update_subject()

In [15]:
interactive_weighted_logits_recontour_viewer(
    root=root,
    subjects=subjects,
    logits_folder="saved_logits_new",
    volume_of_interest="CTVT",
    zoom_fraction=0.30,
    initial_weight=0.50,
    consensus_indices=[3]
)

Loaded subject newAcq_050f229dc2bdb64c with volume of interest 'CTVT'
Image shape: (88, 1024, 1024), Mask shape: (88, 1024, 1024), Uncertainty map shape: (88, 1024, 1024), Ground truth shape: (88, 1024, 1024)
Image spacing (z, y, x): (2.5, 0.46880000829696655, 0.46880000829696655)
